# Lesson 20: The Web Interface

## From CLI to Web App

Last lesson traced how everything connects. Now the **user-facing side** — the web interface.

The web interface is the **primary way to use this product**. Open `http://localhost:5173` in your browser:

- Chat with the team in a clean web UI
- See responses stream in real-time
- Browse and read generated articles in the sidebar
- Delete articles you don't need

The architecture: **FastAPI backend** (AgentOS) + **React frontend** (Vite).

## AgentOS — Backend in 30 Lines

Agno provides **AgentOS** — a wrapper that turns any Agent or Team into a FastAPI server with 50+ endpoints.

```python
from agno.os import AgentOS
from agents.team import team

agent_os = AgentOS(teams=[team])
app = agent_os.get_app()
```

That's it. AgentOS auto-generates:
- `POST /teams/seo-workspace/runs` — send a prompt (supports SSE streaming)
- `GET /health` — health check
- `GET /docs` — interactive Swagger API docs
- Session management, run history, and more

Our `serve.py` adds custom routes for the article storage layer on top.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Let's read the actual serve.py
serve_path = os.path.abspath("../../output/backend/serve.py")
with open(serve_path, "r", encoding="utf-8") as f:
    print(f.read())

## Breaking Down serve.py

The file has 4 parts:

1. **API key validation** — same as chat.py, exits if ANTHROPIC_API_KEY is missing
2. **Custom FastAPI app** — with CORS middleware (allows the React frontend to connect) and 3 article routes
3. **AgentOS wrapper** — `AgentOS(teams=[team], base_app=app)` merges our custom routes with the auto-generated ones
4. **Server start** — `agent_os.serve(app="serve:app", port=7777, reload=True)`

The custom article routes are thin wrappers around `tools.storage` functions:
- `GET /api/articles` → calls `list_articles()`
- `GET /api/articles/{id}` → calls `get_article()`
- `DELETE /api/articles/{id}` → removes from metadata + deletes `.md` file

## The Frontend

The React frontend is in `frontend/` at the project root:

```
frontend/
├── package.json          Dependencies (react, react-markdown, vite)
├── vite.config.js        Dev server proxy → localhost:7777
├── index.html
└── src/
    ├── main.jsx          App entry point
    ├── App.jsx           Layout: sidebar + main area
    ├── api.js            API client (fetch wrappers)
    └── components/
        ├── Chat.jsx      Chat interface with SSE streaming
        ├── ArticleList.jsx  Sidebar article list
        └── ArticleView.jsx  Full article viewer
```

**SSE (Server-Sent Events)** is how the frontend gets streaming responses. Instead of waiting for the full response, the browser receives chunks as they're generated — same experience as ChatGPT.

The Vite dev server proxies `/api` and `/teams` requests to `localhost:7777` (the backend), so the frontend just uses relative URLs.

## Running the Web App

Two terminals:

**Terminal 1 — Backend:**
```bash
python output/backend/serve.py
```
Starts on `http://localhost:7777`. Swagger docs at `/docs`.

**Terminal 2 — Frontend:**
```bash
cd output/frontend
npm install    # first time only
npm run dev
```
Starts on `http://localhost:5173`. Open in browser.

**CLI alternative:**
```bash
python output/backend/chat.py
```
Same team, terminal interface. Useful for development and testing.

In [ ]:
print("To run the web app:\n")
print("  Terminal 1 (backend):  python output/backend/serve.py")
print("  Terminal 2 (frontend): cd output/frontend && npm run dev")
print()
print("Backend: http://localhost:7777")
print("  /docs      -- Swagger API docs")
print("  /health    -- Health check")
print("  /api/articles -- Article list")
print()
print("Frontend: http://localhost:5173")
print("  Chat with the team, browse articles")
print()
print("CLI alternative: python output/backend/chat.py")

## Vibecoding the Backend

Here's how `serve.py` was created — **not by hand, but by describing what we wanted to Claude Code**.

The process:
1. The `CLAUDE.md` file already described the full architecture (agents, tools, storage)
2. We told Claude Code: *"Create a FastAPI backend using AgentOS that serves the team, with custom routes for the article storage layer"*
3. Claude Code read the existing code, understood the patterns, and generated `serve.py`

The key insight: **you don't need to know FastAPI or AgentOS syntax**. You describe what you want in terms of the code that already exists, and Claude Code generates the implementation.

What made this work:
- **CLAUDE.md** gave Claude Code full context about the project
- The **existing code** (team.py, storage.py) was clean and well-organized
- The **prompt** was specific: *which team, which routes, which port*

In [ ]:
# Let's see the CLAUDE.md that gives Claude Code context
claude_md_path = os.path.abspath("../../CLAUDE.md")
with open(claude_md_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"CLAUDE.md ({len(content.splitlines())} lines)\n")
print(content[:3000])
if len(content) > 3000:
    print("\n... (truncated)")

## Quick Note: json.loads() and json.dumps()

You'll see these two functions when working with JSON data in Python:

- `json.loads(text)` — **load string**: converts a JSON text string into a Python dict/list
- `json.dumps(data)` — **dump string**: converts a Python dict/list into a JSON text string

```python
import json

# JSON string -> Python dict
text = '{"title": "SEO Guide", "words": 1500}'
data = json.loads(text)
print(data["title"])  # "SEO Guide"

# Python dict -> JSON string
back_to_text = json.dumps(data)
print(back_to_text)   # '{"title": "SEO Guide", "words": 1500}'
```

The "s" in loads/dumps stands for "string". There's also `json.load(file)` / `json.dump(data, file)` for reading/writing files directly.

## Reading the Production Code

### Mapping: Lesson to Production File

| What you learned | Lesson | Production file |
|---|---|---|
| How LLMs work, prompts, models | 05-07 | (Conceptual — informs all decisions) |
| Agent creation, tools | 08-09 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Structured output (Pydantic) | 10-11 | (Used in AIO analysis) |
| Agent chaining, mini pipeline | 12-13 | (Concepts — replaced by team delegation) |
| Content Writer, Images, AIO | 15-16 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Team and batch processing | 17 | `output/backend/agents/team.py` |
| Local file storage | 18 | `output/backend/tools/storage.py` |
| How everything connects | 19 | All files — the full request flow |
| Web interface (this lesson) | 20 | `output/backend/serve.py` + `frontend/` |

### File structure

```
output/
├── chat.py                     CLI entry point
├── serve.py                    Web backend (AgentOS + article routes)
├── agents/                     Agent definitions (who)
│   ├── __init__.py             Re-exports
│   ├── content_writer.py       Content Writer
│   ├── image_finder.py         Image Finder
│   ├── aio_analyzer.py         AIO Analyzer
│   └── team.py                 Agno Team
└── tools/                      Tool definitions (what)
    ├── __init__.py             Package marker
    ├── storage.py              Local file storage
    ├── search.py               DataForSEO web search
    ├── aio.py                  AIO analysis + credentials
    └── images.py               DataForSEO image search
```

## Summary

The product has **two interfaces** to the same team:

| | Web Interface | CLI |
|---|---|---|
| File | `serve.py` | `chat.py` |
| Framework | FastAPI + AgentOS | Agno cli_app() |
| Access | Browser (localhost:5173) | Terminal |
| Streaming | SSE (Server-Sent Events) | Built-in |
| Articles | Sidebar browser | JSON in terminal |
| Best for | Daily use | Development/testing |

Both use the **same team, same agents, same tools, same storage**.

**Next lesson:** Extending the product — adding new capabilities with vibecoding.

## Exercise

1. Read `output/backend/serve.py` and identify: how many custom routes are there? What does each one do?
2. The frontend's `api.js` file handles SSE streaming. What endpoint does it call for chat? (Hint: look at the `runTeam` function)
3. If you wanted to add a `PUT /api/articles/{id}` route to update article metadata, what storage function would it call?

<details>
<summary>Click to reveal answers</summary>

1. **3 custom routes:** `GET /api/articles` (list all), `GET /api/articles/{id}` (get one), `DELETE /api/articles/{id}` (delete one).

2. **Chat endpoint:** `POST /teams/seo-workspace/runs` with `stream: true` in the request body. The frontend reads SSE events from the response.

3. **PUT route:** It would call `update_article_content(article_id, article_markdown)` from `tools.storage`. You'd also need to update the metadata fields in `articles.json` directly or add a new storage function.
</details>